# broadcast-source-fanout — worked example 1: Emulate dist.broadcast with a send/recv fan-out loop on one process

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-source-fanout`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A broadcast collective copies one source rank's tensor to every other rank. The textbook implementation has the source rank `dist.send` its buffer to each other rank in a loop, while every non-source rank does a single `dist.recv` from `src` and `copy_`s the bytes into its local tensor. We can emulate this fan-out deterministically in a single process by simulating each rank's local buffer in a Python dict.

## Worked solution

**Goal.** Mimic `dist.broadcast(tensor, src)` semantics without a real process group, so we can see the fan-out logic clearly.

1. **Set up per-rank buffers.** Each rank owns its own tensor. We model `world_size` ranks as a dict `buffers[r]`. Only the source rank holds meaningful data; the others are zero-filled placeholders that will be overwritten — exactly like uninitialized `recv` buffers.
2. **Source fan-out (the `send` loop).** The source rank iterates over every other rank `r != src` and pushes a *copy* of its payload. In real torch this is `dist.send(tensor, dst=r)`; here it is a `.clone()` into `buffers[r]`. Cloning matters: a real `recv` writes into a *separate* allocation on the destination, so mutating one rank's buffer later must not alias another's.
3. **Receiver copy_ (the `recv`).** Non-source ranks overwrite their local buffer in place with `copy_`. We model this as assigning the cloned tensor, matching `recv_buf.copy_(incoming)` semantics.
4. **Why it is correct.** After the loop every rank holds an *equal-valued but independent* tensor. We verify independence by mutating the source afterward and confirming the receivers are unchanged — proving we copied, not aliased.
5. **Result.** Stacking all rank buffers yields a `(world_size, *shape)` tensor whose rows are all identical to the source payload.

In [ ]:
import torch as t
from torch import Tensor

def manual_broadcast(payload: Tensor, src: int, world_size: int):
    # Each rank's local buffer. Non-source ranks start as zeros (uninitialized recv buffers).
    buffers = {r: t.zeros_like(payload) for r in range(world_size)}
    buffers[src] = payload.clone()
    # Source rank loops dist.send to every other rank; receivers copy_ into local buffer.
    for r in range(world_size):
        if r == src:
            continue
        incoming = buffers[src].clone()   # models the wire copy of send
        buffers[r].copy_(incoming)        # models recv_buf.copy_(incoming)
    return t.stack([buffers[r] for r in range(world_size)], dim=0)

t.manual_seed(0)
payload = t.randn(3)
out = manual_broadcast(payload, src=2, world_size=4)
print("stacked shape:", tuple(out.shape))
print("all rows equal source:", bool(t.allclose(out, payload.expand(4, 3))))
# Independence check: mutate source buffer copy, receivers already finalized.
print("row 0 matches src row 2:", bool(t.allclose(out[0], out[2])))